### Update log

last update: 5/12/2025 11:11pm
* added progress bars
* removing redundancies now goes from the end by default
* made dynamic programming in recursive path order only engage for words longer than 20 letters, also fixed a bug in recursive path order that prevented it from actually using alphabet_order
* added code for B(2,3) and B(2,4)
* B(2,4) should take 2-4 hours to run

last update: 5/7/2025 9:38pm
* added naive dynamic programming implementation of recursive_path_order
* it's a bit faster than recursion
* dynamic programming implementation now used by default in recursive_path_order

last update: 5/6/2025 1:04pm
* added positive first alphabet order and all 6 respective options for word order:
  - (shortlex, recurseive)x(default, chenadec, positive first)
* some of the 6 option resolve every example, including braid group

## Knuth-Bendix implementation

In [1]:
from tqdm import tqdm

### Rule Definition

In [2]:
rules = []
''' rules stored as [length of LHS, LHS, RHS] '''
''' for example: '''
''' 1234->567 is stored as [4, 1,2,3,4, 5,6,7] '''
''' x_1 x_2 x_1^{-1} -> x_2^2 stored as [3, 1,2,-1, 2,2] '''
''' together: rules = [ [4, 1,2,3,4, 5,6,7] , [3, 1,2,-1, 2,2]] '''

def rule_left(rule):
  '''extracts lhs of rule'''
  return rule[1:rule[0]+1]

def rule_right(rule):
  '''extracts rhs of rule'''
  return rule[rule[0]+1:]

### Order Definitions

In [3]:
def alphabet_order(a,b):
  ''' x_1 < x_1^{-1} < x_2 < x_2^{-1} < .... '''
  ''' 1 means LHS > RHS, -1 means LHS < RHS, 0 means LHS = RHS '''
  if abs(a) > abs(b):
    return 1
  elif abs(a) < abs(b):
    return -1
  elif a<b:
    return -1
  elif a>b:
    return 1
  else:
    return 0

def alphabet_order_positive_first(a,b):
  ''' x_1 < x_2 < ... < x_n < x_1^{-1} < x_2^{-1} < ... < x_n^{-1} '''
  ''' 1 means LHS > RHS, -1 means LHS < RHS, 0 means LHS = RHS '''
  if a > 0 and b < 0:
    return -1
  if a < 0 and b > 0:
    return 1
  if a > 0 and b > 0:
    if a > b:
      return 1
    elif a < b:
      return -1
    else:
      return 0
  if a < 0 and b < 0:
    if a < b:
      return 1
    elif a > b:
      return -1
    else:
      return 0

def chenadec_order(a,b):
  ''' x_2p > x_2p^{-1} > x_{2p-2} > .... > x_2^{-1} > x_1 > x_1^{-1} > ... > x_{2p-1}^{-1}'''
  ''' 1 means LHS > RHS, -1 means LHS < RHS, 0 means LHS = RHS '''
  if a%2 == 0 and b%2 == 1:
    return 1
  elif a%2 == 1 and b%2 == 0:
    return -1
  elif a%2 == 0 and b%2 == 0:
    if abs(a) > abs(b):
      return 1
    elif abs(a) < abs(b):
      return -1
    elif a > b:
      return 1
    elif a < b:
      return -1
    else:
      return 0
  elif a%2 == 1 and b%2 == 1:
    if abs(a) < abs(b):
      return 1
    elif abs(a) > abs(b):
      return -1
    elif a > b:
      return 1
    elif a < b:
      return -1
    else:
      return 0
  return 0

def shortlex(word1,word2,alphabet_order=alphabet_order):
  ''' 1 means LHS > RHS, -1 means LHS < RHS, 0 means LHS = RHS '''
  if len(word1) > len(word2):
    return 1
  elif len(word1) < len(word2):
    return -1
  else:
    for i in range(len(word1)):
      if alphabet_order(word1[i],word2[i]) == 1:
        return 1
      elif alphabet_order(word1[i],word2[i]) == -1:
        return -1
    return 0

def shortlex_with_chenadec(a,b):
  return shortlex(a,b,alphabet_order=chenadec_order)

def shortlex_with_positive_first(a,b):
  return shortlex(a,b,alphabet_order=alphabet_order_positive_first)

def lex(word1,word2,alphabet_order=alphabet_order):
  ''' may lead to infinite reduction chain, do not use unless you know what you are doing '''
  ''' 1 means LHS > RHS, -1 means LHS < RHS, 0 means LHS = RHS '''
  for i in range(min(len(word1),len(word2))):
    if alphabet_order(word1[i],word2[i]) == 1:
      return 1
    elif alphabet_order(word1[i],word2[i]) == -1:
      return -1
  if len(word1) > len(word2):
    return 1
  elif len(word1) < len(word2):
    return -1
  # else:
  return 0

def recursive_path_order(word1,word2,alphabet_order=alphabet_order, use_dynprog = True):
  ''' as in Hermiller-Shapiro paper '''
  ''' 1 means LHS > RHS, -1 means LHS < RHS, 0 means LHS = RHS '''
  ''' by default uses naive dynamic programming implementation it's 10-30% faster on words up 200 letters '''
  if use_dynprog and (len(word1) > 20 or len(word2) > 20):
    return recursive_path_order_dyn_prog(word1,word2,alphabet_order=alphabet_order)

  if len(word1) == 1 and len(word2) == 1:
    return alphabet_order(word1[0],word2[0])
  if len(word1) > 0 and len(word2) == 0:
      return 1
  if len(word1) == 0 and len(word2) > 0:
      return -1
  if word1[0] == word2[0]:
    # print("case 1")
    return recursive_path_order(word1[1:],word2[1:],alphabet_order)
  if alphabet_order(word1[0],word2[0]) == 1 and recursive_path_order(word1,word2[1:],alphabet_order) == 1:
    # print("case 2")
    return 1
  # print("case 3")
  ord = recursive_path_order(word1[1:],word2,alphabet_order)
  if ord == 1 or ord == 0:
    return 1
  return -1

def recursive_path_order_dyn_prog(word1, word2, alphabet_order=alphabet_order):
  """Iterative implementation of the recursive path order.

  Args:
    word1: The first word.
    word2: The second word.
    alphabet_order: The alphabet order function.

  Returns:
    1 if word1 > word2, -1 if word1 < word2, 0 if word1 == word2.
  """
  if len(word1) == 1 and len(word2) == 1:
    return alphabet_order(word1[0],word2[0])
  if len(word1) > 0 and len(word2) == 0:
      return 1
  if len(word1) == 0 and len(word2) > 0:
      return -1

  dynamic_prog_table = [[0 for _ in range(len(word2)+1)] for _ in range(len(word1)+1)]

  len1=len(word1)
  len2=len(word2)
  for k in range(len2):
    dynamic_prog_table[len1][k] = -1
  for k in range(len1):
    dynamic_prog_table[k][len2] = 1
    dynamic_prog_table[len1][len2] = 0
  # for ii in range(len1+1):
    # print(dynamic_prog_table[len1-ii])
  # print("-------")
  for diag in reversed(range(len1+len2-1)): # i+j=diag
    # print("Diag =",diag)
    start_i = 0 if diag-len2+1 < 0 else diag-len2+1
    end_j = 0 if diag-len1+1 < 0 else diag-len1+1
    for i in range(start_i, diag-end_j+1):
      j = diag-i
      if word1[i] == word2[j]:
        dynamic_prog_table[i][j] = dynamic_prog_table[i+1][j+1]
      elif alphabet_order(word1[i], word2[j]) == 1 and dynamic_prog_table[i][j+1] == 1:
        dynamic_prog_table[i][j] = 1
      # elif word1[i] < word2[j] and dynamic_prog_table[i+1][j] == -1:
        # dynamic_prog_table[i][j] = -1
      elif dynamic_prog_table[i+1][j] >= 0:
        dynamic_prog_table[i][j] = 1
      # elif dynamic_prog_table[i][j+1] >= 0:
        # dynamic_prog_table[i][j] = -1
      else:
        dynamic_prog_table[i][j] = -1
      # for ii in range(len1+1):
        # print(dynamic_prog_table[len1-ii])
      # print("-------")
  return dynamic_prog_table[0][0]

def recursive_path_order_with_chenadec(a,b):
  return recursive_path_order(a,b,alphabet_order=chenadec_order)

def recursive_path_order_with_positive_first(a,b):
  return recursive_path_order(a,b,alphabet_order=alphabet_order_positive_first)

def wreath_order(word1,word2,alphabet_order=alphabet_order):
  ''' not implemented '''
  return 0

def reorder_rules(rules, order):
  ''' for each rule u->v, rewrite it as u->v or v->u according to order '''
  for i in range(len(rules)):
    if order(rule_left(rules[i]),rule_right(rules[i])) == -1:
      rules[i] = [len(rule_right(rules[i]))] + rule_right(rules[i]) + rule_left(rules[i])

# shortlex([1,2],[1,2,1],alphabet_order)


### Rule Application

In [4]:
def apply_rule(word, rule, place=0):
  ''' stupidly searches for lhs of rule starting from place and replaces with rhs '''
  ''' there are faster ways to implement this but this at least works '''
  ''' returns new word, or unchanged word if rule does not apply '''
  lhs = rule_left(rule)
  rhs = rule_right(rule)
  for i in range(place,len(word) - len(lhs) + 1):
    if word[i : i + len(lhs)] == lhs:
      word = word[:i] + rhs + word[i+len(lhs):]
  return word

#apply_rule([1,2,3,4,5],[2,3,4,-1,-1,-1])

def reduce(word, rules, skip_rule = -1, print_progress = False):
  ''' reduces the word, choosing first applicable place of first applicable rule '''
  ''' skip_rule is the rule to skip, needed for redundancy check in K-B algorithm '''
  ''' returns reduces word '''
  while True:
    if print_progress:
      print("new round...")
    starting_word = word
    for i in range(len(rules)):
      if i != skip_rule:
        word = apply_rule(word, rules[i])
    if word == starting_word:
      break
  return word

### Critical Pair Definition and Resolution

In [7]:
crit_pairs = []
# for overlapping rules u_1 v -> w_1, v u_2 -> w_2, crit pair is the result of applying each rule to u_1 v u_2:
# [w_1 u_2, u_1 w_2]
# for inculsive rules v -> w_1, u v u' -> w_2, crit pair is the result of applying each rule to u v u':
# [w_2, u w_1 u']
# (u, u' allowed to be empty in the code below, u_1, u_2 shouldn't be empty unless I messed up)


def make_crit_pairs(rules, crit_pairs, reduce_immediately = True, start_at_rule=0, max_crit_length=0, reset_pairs = True, \
                    print_progress=False, print_progress_percentage=False):
  ''' returns list of crit pairs for the given list of rules'''
  ''' if reduce_immediately == True, the crit pairs resolved by existing rules are skipped '''
  ''' if start_at_rule > 0, only crit pairs involving at least one rule with number >= start_at_rule will be processed '''
  ''' '''
  if reset_pairs:
    crit_pairs.clear()

  #lhs overlap
  if print_progress_percentage:
    print("Building crit pairs - overlap:")
  i_range = tqdm(range(len(rules))) if print_progress_percentage else range(len(rules))
  for i in i_range:
    for j in range(len(rules)):
      if i < start_at_rule and j < start_at_rule:
        continue
      lhs1 = rule_left(rules[i])
      lhs2 = rule_left(rules[j])
      #if print_progress:
      #  print(lhs1,lhs2)
      for k in range(1,min(len(lhs2),len(lhs1))): # checking for overlap of size k
        if (max_crit_length==0 or len(lhs1)+len(lhs2)-k <= max_crit_length) and lhs1[-k:] == lhs2[:k]:
          if reduce_immediately:
            reduced_word1 = reduce(rule_right(rules[i])+lhs2[k:], rules)
            reduced_word2 = reduce(lhs1[:-k]+rule_right(rules[j]), rules)
            if print_progress:
                print("Overlap:", lhs1[:-k] + lhs1[-k:] + lhs2[k:], "with lhs:", lhs1, lhs2, "Reduced words:", reduced_word1, reduced_word2)
            if reduced_word1 != reduced_word2:
              crit_pairs.append([reduced_word1, reduced_word2])
          else:
            if print_progress:
              print("Overlap:", lhs1[:-k] + lhs1[-k:] + lhs2[k:], "with lhs:", lhs1, lhs2, "Words:", rule_right(rules[i])+lhs2[k:], lhs1[:-k]+rule_right(rules[j]))
            crit_pairs.append([rule_right(rules[i])+lhs2[k:] , lhs1[:-k]+rule_right(rules[j])])

  #lhs inclusion
  if print_progress_percentage:
    print("Building crit pairs - inclusion:")
  i_range = tqdm(range(len(rules))) if print_progress_percentage else range(len(rules))
  for i in i_range:
    lhs1 = rule_left(rules[i])
    for j in range(start_at_rule,len(rules)):
      if i == j:
        continue
      lhs2 = rule_left(rules[j])
      if (max_crit_length==0 or len(lhs2) <= max_crit_length) and len(lhs2) >= len(lhs1):
        for k in range(len(lhs2)-len(lhs1)+1):
          if lhs2[k:k+len(lhs1)] == lhs1:
            if reduce_immediately:
              reduced_word1 = reduce(lhs2[:k]+rule_right(rules[i])+lhs2[k+len(lhs1):], rules)
              reduced_word2 = reduce(rule_right(rules[j]), rules)
              if print_progress:
                print("Inclusion:", lhs2, "with lhs:", lhs1, lhs2, "Reduced words:", reduced_word1, reduced_word2)
              if reduced_word1 != reduced_word2:
                crit_pairs.append([reduced_word1, reduced_word2])
            else:
              if print_progress:
                print("Inclusion:", lhs2, "with lhs:", lhs1, lhs2, "Words:", lhs2[:k]+rule_right(rules[i])+lhs2[k+len(lhs1):], rule_right(rules[j]))
              crit_pairs.append([lhs2[:k]+rule_right(rules[i])+lhs2[k+len(lhs1):], rule_right(rules[j])])
      elif (i < start_at_rule and j >= start_at_rule) and (max_crit_length==0 or len(lhs1) <= max_crit_length) and len(lhs2) < len(lhs1):
        # only pairs (i,j) where i < start_at_rule and j >= start_at_rule need to be swapped
        # pairs where both i,j >= start_at_rule will be encountered as (i,j) and as (j,i) anyway
        for k in range(len(lhs1)-len(lhs2)+1):
          if lhs1[k:k+len(lhs2)] == lhs2:
            if reduce_immediately:
              reduced_word1 = reduce(lhs1[:k]+rule_right(rules[j])+lhs1[k+len(lhs2):], rules)
              reduced_word2 = reduce(rule_right(rules[i]), rules)
              if print_progress:
                print("Inclusion:", lhs1, "with lhs:", lhs2, lhs1, "Reduced words:", reduced_word1, reduced_word2)
              if reduced_word1 != reduced_word2:
                crit_pairs.append([reduced_word1, reduced_word2])
            else:
              if print_progress:
                print("Inclusion:", lhs1, "with lhs:", lhs2, lhs1, "Words:", lhs1[:k]+rule_right(rules[j])+lhs1[k+len(lhs2):], rule_right(rules[i]))
              crit_pairs.append([lhs1[:k]+rule_right(rules[j])+lhs1[k+len(lhs2):], rule_right(rules[i])])
  return crit_pairs

def resolve_crit_pair(rules, crit_pairs, i, order=shortlex, print_progress = False):
  ''' resolves crit pair number i '''
  ''' reduces both words in crit pair i using existing rules '''
  ''' if results are not equal, adds a rule word1 -> word2 or word2 -> word1 depending on order '''
  ''' reduction is redundant if crit_pairs was obtained from make_crit_pairs() with reduce_immediately = True '''
  if print_progress:
    print("Resolving ", crit_pairs[i])
  word1 = reduce(crit_pairs[i][0],rules, print_progress=print_progress)
  word2 = reduce(crit_pairs[i][1],rules, print_progress=print_progress)
  if word1 != word2:
    if order(word1,word2) == 1:
      rules.append([len(word1)] + word1 + word2)
    else:
      rules.append([len(word2)] + word2 + word1)
    crit_pairs.remove(crit_pairs[i])
    if print_progress:
      print("Added rule", rules[-1])
    return 1
  crit_pairs.remove(crit_pairs[i])
  if print_progress:
      print("No rule added")
  return 0

def resolve_all_crit_pairs(rules, crit_pairs = [], order=shortlex, reduce_crit_immediately = True, go_from_the_end=True, print_progress = False, print_crit_pairs_progress = False):
  ''' Resolves crit_pairs going from the end or the beginning of the list according to go_from_the_end'''
  ''' Returns True if any new rules were added '''
  if crit_pairs == []:
    crit_pairs = make_crit_pairs(rules, crit_pairs, reduce_immediately=reduce_crit_immediately, print_progress=print_crit_pairs_progress)
  rule_counter = 0
  if print_progress:
    print("Resolving crit pairs...")
    pbar = tqdm(total = len(crit_pairs))
  while len(crit_pairs) > 0:
    rule_number_to_resolve = -1 if go_from_the_end else 0
    rule_counter+=resolve_crit_pair(rules, crit_pairs, rule_number_to_resolve, order=order)
    if print_progress:
        # print("Crit pairs left to resolve:", len(crit_pairs), "     ", end="\r")
        pbar.update(1)
  if print_progress:
      # print("                                                           ", end="\r")
      pbar.close()

  if rule_counter > 0:
    if print_progress:
      print("Added", rule_counter, "new rules")
    return True
  return False

### Confluence Check

In [8]:
def check_confluence(rules,crit_pairs=[], max_crit_length = 0, erase_pair_list = True, print_progress = False):
  ''' checks if the rewriting system is confluent'''
  ''' redundant if knuth_bendix() converged '''
  ''' will clear list of crit pairs after running if erase_pair_list == True (default) '''
  if crit_pairs == []:
    crit_pairs = make_crit_pairs(rules, crit_pairs, max_crit_length=max_crit_length,print_progress=print_progress)
  for i in range(len(crit_pairs)):
    if print_progress:
      print(crit_pairs[i])
    word1 = reduce(crit_pairs[i][0],rules, print_progress=False)
    word2 = reduce(crit_pairs[i][1],rules, print_progress=False)
    if word1 != word2:
      if print_progress:
        print("Confluence fail at", crit_pairs[i][0], crit_pairs[i][1], "-->", word1, word2)
      if erase_pair_list:
        crit_pairs.clear()
      return False
  if erase_pair_list:
        crit_pairs.clear()
  return True

### Rule Reduction Shortcut

In [9]:
def shortcut(rules, rule_number):
  ''' Reduces RHS of rule number rule_number '''
  ''' Returns True if a shortcut was made '''
  rhs=rule_right(rules[rule_number])
  lhs=rule_left(rules[rule_number])
  reduced_rhs = reduce(rhs,rules)
  if reduced_rhs != rhs:
   # rules[rule_number] = [len(lhs)] + lhs + reduced_rhs
    rules[rule_number][len(lhs)+1:] = reduced_rhs
    return True
  return False

def make_all_shortcuts(rules):
  ''' Reduces RHS of all rules '''
  ''' Returns True if any shortcuts were made '''
  changes_made = False
  for i in range(len(rules)):
    changes_made = shortcut(rules,i) or changes_made
  return changes_made


### Rule Reduction

In [11]:
def is_rule_redundant(rules, rule_number):
  ''' Checks if rule number rule_number is redundant '''
  ''' by reducing its LHS and RHS by all other rules '''
  ''' Returns True if rule is redundant '''
  if reduce(rule_left(rules[rule_number]), rules, skip_rule=rule_number) == \
     reduce(rule_right(rules[rule_number]), rules, skip_rule=rule_number):
    return True
  return False

def eliminate_redundancy(rules,start_at_rule=0, go_from_the_end=True):
  ''' Eliminates redundant rules '''
  ''' Returns True if any rules were eliminated and '''
  ''' the new number of first new rule to use when making crit pairs '''
  redundancies_were_present = False
  i=0
  new_first_rule=start_at_rule
  if go_from_the_end:
    while i < len(rules):
      # each iteration of cycle either i increases by 1, or len(rules) decreases by 1 '''
      if is_rule_redundant(rules,len(rules)-1-i):
        redundancies_were_present = True
        if len(rules)-1-i < new_first_rule:
          new_first_rule-=1 # old rule was deleted, need to shift first new rule number
        del rules[-1-i]
      else:
        i+=1
  else:
    while i < len(rules):
      # each iteration of cycle either i increases by 1, or len(rules) decreases by 1 '''
      if is_rule_redundant(rules,i):
        redundancies_were_present = True
        rules.remove(rules[i])
        if i < new_first_rule:
          new_first_rule-=1 # old rule was deleted, need to shift first new rule number
      else:
        i+=1
  return redundancies_were_present, new_first_rule

In [12]:
def knuth_bendix(rules, order=shortlex, reduce_crit_immediately = True, max_rounds=-1, recheck_old_rules = False, \
                 print_progress = False, print_crit_pairs_progress = False):
  ''' Runs Knuth-Bendix algorithm on the given list of rules '''
  ''' Follows Algorithm 6.2.11 in Word Processing in Groups by Epstein at al. (Section 6.2) '''
  ''' Returns -1 if terminated by reaching max_rounds; or number of rounds if terminated by attaining confluence '''
  changes_made = True
  rounds=0
  first_new_rule=0
  crit_pairs = []
  while changes_made:
    rounds=rounds+1
    if max_rounds >= 0 and rounds > max_rounds:
      print("Max rounds reached:", max_rounds)
      print("Making shortcuts and removing redundancies, then stopping.")
      # return -1
    if print_progress:
      print("Round", rounds)
    changes_made = False
    shortcuts_made = False
    new_rules_added = False
    redundancies_eliminated = False

    # make all shortcuts
    shortcuts_made = make_all_shortcuts(rules)
    if print_progress and shortcuts_made:
      print("Shortcuts made.", len(rules), "rules total.")

    # remove redundant rules
    redundancies_eliminated, first_new_rule = eliminate_redundancy(rules, first_new_rule)
    if print_progress and redundancies_eliminated:
      print("Redundancies eliminated.", len(rules), "rules total.")
      # print(len(rules), "rules total:\n", rules)

    if max_rounds >= 0 and rounds > max_rounds:
      print("Stopping because max rounds reached:", max_rounds)
      return -1

    # make crit pairs only using at least one new rule
    assert(crit_pairs == [])
    if recheck_old_rules:
      make_crit_pairs(rules, crit_pairs, reduce_immediately=reduce_crit_immediately, start_at_rule=0, \
                      print_progress=print_crit_pairs_progress, print_progress_percentage = print_progress)
    else:
      make_crit_pairs(rules, crit_pairs, reduce_immediately=reduce_crit_immediately, start_at_rule=first_new_rule, \
                      print_progress=print_crit_pairs_progress, print_progress_percentage = print_progress)

    if print_progress:
      print("Crit pairs:", len(crit_pairs))

    # all rules become old
    first_new_rule = len(rules)

    # add new rules to resolve all crit pairs
    new_rules_added = resolve_all_crit_pairs(rules, crit_pairs=crit_pairs, order=order, reduce_crit_immediately=reduce_crit_immediately, \
                                             print_progress=print_progress)
    if print_progress and new_rules_added:
      print("New rules added.", len(rules), "rules total.")
    changes_made = shortcuts_made or redundancies_eliminated or new_rules_added
    if print_progress:
      # print(len(rules), "rules total:\n", rules)
        print(len(rules), "rules total at the end of round", rounds)
  return rounds

In [13]:
def group_inverse(word):
  neg_word = [-x for x in word]
  return neg_word[::-1]

def symmetrize_group_rule(rules,i,order=shortlex):
  ''' symmetrizes rule number i assuming group structure: '''
  ''' for a rule u=v, it computes r=uv^{-1} and iterates over all cyclic shifts and their inverses r' of r: '''
  ''' for each r', it adds all rules obtained from writing r'as u' = v' in all possible ways '''
  ''' rule of length n adds 2n*n rules '''
  word = rule_left(rules[i]) + group_inverse(rule_right(rules[i]))
  # print(word)

  for j in range(len(word)):
    shifted_word = word[j:] + word[:j]
    # print("shifted word = ", shifted_word)
    for i in range(len(word)):
      word1 = shifted_word[:i]
      word2 = group_inverse(shifted_word[i:])
      if order(word1,word2) == 1:
        rules.append([len(word1)] + word1 + word2)
        # print("added rule:", [len(word1)] + word1 + word2)
      elif order(word1,word2) == -1:
        rules.append([len(word2)] + word2 + word1)
        # print("added rule:", [len(word2)] + word2 + word1)
      # else:
        # print("error")

  # print(len(rules), "before inverse:", rules)
  word = group_inverse(word)
  # print("inverse word:", word)
  for j in range(len(word)):
    shifted_word = word[j:] + word[:j]
    for i in range(len(word)):
      word1 = shifted_word[:i]
      word2 = group_inverse(shifted_word[i:])
      if order(word1,word2) == 1:
        rules.append([len(word1)] + word1 + word2)
      elif order(word1,word2) == -1:
        rules.append([len(word2)] + word2 + word1)
      # else:
        # print("error")

def add_free_group_rules(rules):
  ''' adds rules of the form x x^{-1} -> 1, x^{-1} x -> 1 for all x as high as encountered in rules '''
  max_x = 1
  for i in range(len(rules)):
    max_tmp = max(rules[i][1:])
    min_tmp = min(rules[i][1:])
    if max_tmp > max_x:
      max_x = max_tmp
    if -min_tmp > max_x:
      max_x = -min_tmp
  for i in reversed(range(1,max_x+1)):
    rules.insert(0,[2,-i,i])
    rules.insert(0,[2,i,-i])

### Examples

In [14]:
# Abelian group on a, b
rules = [[4,1,2,-1,-2]] #Z^2
add_free_group_rules(rules)
print("Starting rules:", rules)
rounds = knuth_bendix(rules, order=shortlex, reduce_crit_immediately=True, max_rounds=10, recheck_old_rules = True, print_progress=False, print_crit_pairs_progress = False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")

# Abelian group on a, b
rules = [[4,1,2,-1,-2]] #Z^2
add_free_group_rules(rules)
print("Starting rules:", rules)
rounds = knuth_bendix(rules, order=shortlex, reduce_crit_immediately=True, max_rounds=10, recheck_old_rules = False, print_progress=False, print_crit_pairs_progress = False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")


Starting rules: [[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [4, 1, 2, -1, -2]]
Finished in 6 rounds.
Total number of rules: 8
[[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 2, -1, -1, 2], [2, 2, 1, 1, 2], [2, -2, -1, -1, -2], [2, -2, 1, 1, -2]]
Confluence: True
-------
Starting rules: [[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [4, 1, 2, -1, -2]]
Finished in 6 rounds.
Total number of rules: 8
[[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 2, -1, -1, 2], [2, 2, 1, 1, 2], [2, -2, -1, -1, -2], [2, -2, 1, 1, -2]]
Confluence: True
-------


In [15]:
# Abelian group on a, b
rules = [[4,1,2,-1,-2]] #Z^2
add_free_group_rules(rules)
print("Starting rules:", rules)
rounds = knuth_bendix(rules, order=shortlex, reduce_crit_immediately=True, max_rounds=10, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")

# Abelian group on a, b but the rule is not order-reducing:
rules = [[2,1,2,2,1]] #Z^2
add_free_group_rules(rules)
print("Starting rules:", rules)
rounds = knuth_bendix(rules, order=shortlex, reduce_crit_immediately=True, max_rounds=20, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")

# Abelian group on a, b but the rule is not order-reducing
# but we reorder rules according to shortlex first lol:
rules = [[2,1,2,2,1]] #Z^2
add_free_group_rules(rules)
reorder_rules(rules,shortlex)
print("Starting rules after reordering:", rules)
rounds = knuth_bendix(rules, order=shortlex, reduce_crit_immediately=True, max_rounds=20, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")

# Abelian group on a, b but the rule is order-increasing
# but we reorder rules according to shortlex first lol:
rules = [[1,1,2,1,-2]] #Z^2
add_free_group_rules(rules)
reorder_rules(rules,shortlex)
print("Starting rules after reordering:", rules)
rounds = knuth_bendix(rules, order=shortlex, reduce_crit_immediately=True, max_rounds=20, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")

# Abelian group on a, b but the rule is written as aba^{-1}=b:
rules = [[3,1,2,-1,2]] #Z^2
add_free_group_rules(rules)
print("Starting rules:", rules)
rounds = knuth_bendix(rules, order=shortlex, reduce_crit_immediately=True, max_rounds=20, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")

# Abelian group on a, b, c
rules = [[4,1,2,-1,-2], [4,1,3,-1,-3], [4,2,3,-2,-3]] #Z^3
add_free_group_rules(rules)
print("Starting rules:", rules)
rounds = knuth_bendix(rules, order=shortlex, reduce_crit_immediately=True, max_rounds=20, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("Reduction example:")
print([-3,-1,-2,3,2,1,1,1,3,2,2,-1,-3,-3], "->", reduce([-3,-1,-2,3,2,1,1,1,3,2,2,-1,-3,-3],rules))
print("-------")


Starting rules: [[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [4, 1, 2, -1, -2]]
Finished in 6 rounds.
Total number of rules: 8
[[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 2, -1, -1, 2], [2, 2, 1, 1, 2], [2, -2, -1, -1, -2], [2, -2, 1, 1, -2]]
Confluence: True
-------
Starting rules: [[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 1, 2, 2, 1]]
Max rounds reached: 20
Making shortcuts and removing redundancies, then stopping.
Stopping because max rounds reached: 20
Terminated by reaching max_rounds.
Total number of rules: 48
[[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 1, 2, 2, 1], [3, 2, 1, -2, 1], [3, -1, 2, 1, 2], [4, -1, 2, 2, 1, 2, 2], [2, 2, -1, -1, 2], [4, 2, 1, 1, -2, 1, 1], [2, -2, 1, 1, -2], [5, 2, 1, 1, 1, -2, 1, 1, 1], [5, -1, 2, 2, 2, 1, 2, 2, 2], [2, -2, -1, -1, -2], [6, -1, 2, 2, 2, 2, 1, 2, 2, 2, 2], [6, 2, 1, 1, 1, 1, -2, 1, 1, 1, 1], [7, 2, 1, 1, 1, 1, 1, -2, 1, 1, 1, 1, 1], [7, -1, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2], [8, -1, 2, 2, 2, 2, 2, 2, 1, 

In [16]:
# RAAG group on a, b, c with [a,b]=[b,c]=1
rules = [[4,1,2,-1,-2],[4,2,3,-2,-3]]
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying shortlex:")
rounds = knuth_bendix(rules, order=shortlex, reduce_crit_immediately=True, max_rounds=10, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")

rules = [[4,1,2,-1,-2],[4,2,3,-2,-3]]
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying recursive path ordering:")
rounds = knuth_bendix(rules, order=recursive_path_order, reduce_crit_immediately=True, max_rounds=10, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")

rules = [[4,1,2,-1,-2],[4,2,3,-2,-3]]
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying shortlex with Chenadec alphabet order:")
rounds = knuth_bendix(rules, order=shortlex_with_chenadec, reduce_crit_immediately=True, max_rounds=10, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")




Starting rules: [[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 3, -3], [2, -3, 3], [4, 1, 2, -1, -2], [4, 2, 3, -2, -3]]
Trying shortlex:
Max rounds reached: 10
Making shortcuts and removing redundancies, then stopping.
Stopping because max rounds reached: 10
Terminated by reaching max_rounds.
Total number of rules: 73
[[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 3, -3], [2, -3, 3], [2, 2, -1, -1, 2], [2, 3, -2, -2, 3], [2, 2, 1, 1, 2], [2, 3, 2, 2, 3], [2, -3, -2, -2, -3], [2, -2, -1, -1, -2], [3, -3, -1, -2, -2, -3, -1], [3, 3, 1, 2, 2, 3, 1], [3, 3, -1, 2, 2, 3, -1], [3, 3, -1, -2, -2, 3, -1], [2, -2, 1, 1, -2], [2, -3, 2, 2, -3], [4, 3, -1, -1, -2, -2, 3, -1, -1], [4, 3, -1, -1, 2, 2, 3, -1, -1], [4, 3, 1, 1, 2, 2, 3, 1, 1], [4, -3, -1, -1, -2, -2, -3, -1, -1], [5, -3, -1, -1, -1, -2, -2, -3, -1, -1, -1], [3, -3, 1, -2, -2, -3, 1], [5, 3, 1, 1, 1, 2, 2, 3, 1, 1, 1], [5, 3, -1, -1, -1, 2, 2, 3, -1, -1, -1], [5, 3, -1, -1, -1, -2, -2, 3, -1, -1, -1], [3, -3, 1, 2, 2, -3

In [17]:
# RAAG group on a, b, c with [a,b]=[a,c]=1
rules = [[4,1,2,-1,-2],[4,1,3,-1,-3]]
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying shortlex:")
rounds = knuth_bendix(rules, order=shortlex, reduce_crit_immediately=True, max_rounds=10, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")

rules = [[4,1,2,-1,-2],[4,1,3,-1,-3]]
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying recursive path ordering:")
rounds = knuth_bendix(rules, order=recursive_path_order, reduce_crit_immediately=True, max_rounds=10, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")

rules = [[4,1,2,-1,-2],[4,1,3,-1,-3]]
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying shortlex with Chenadec alphabet order:")
rounds = knuth_bendix(rules, order=shortlex_with_chenadec, reduce_crit_immediately=True, max_rounds=10, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")

Starting rules: [[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 3, -3], [2, -3, 3], [4, 1, 2, -1, -2], [4, 1, 3, -1, -3]]
Trying shortlex:
Finished in 6 rounds.
Total number of rules: 14
[[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 3, -3], [2, -3, 3], [2, 2, -1, -1, 2], [2, 3, -1, -1, 3], [2, 2, 1, 1, 2], [2, 3, 1, 1, 3], [2, -3, -1, -1, -3], [2, -2, -1, -1, -2], [2, -2, 1, 1, -2], [2, -3, 1, 1, -3]]
Confluence: True
-------
Starting rules: [[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 3, -3], [2, -3, 3], [4, 1, 2, -1, -2], [4, 1, 3, -1, -3]]
Trying recursive path ordering:
Finished in 6 rounds.
Total number of rules: 14
[[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 3, -3], [2, -3, 3], [2, 2, -1, -1, 2], [2, 3, -1, -1, 3], [2, 2, 1, 1, 2], [2, 3, 1, 1, 3], [2, -3, -1, -1, -3], [2, -2, -1, -1, -2], [2, -2, 1, 1, -2], [2, -3, 1, 1, -3]]
Confluence: True
-------
Starting rules: [[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 3, -3], [2, -3, 3], [4, 1, 2, -1

In [18]:
# braid group < a,b,c | a^3 = b^2 = c >
rules = [[3,1,1,1,2,2], [3,1,1,1,3]]
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying shortlex:")
rounds = knuth_bendix(rules, order=shortlex, reduce_crit_immediately=True, max_rounds=7, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")

# braid group < a,b,c | a^3 = b^2 = c >
rules = [[3,1,1,1,2,2], [3,1,1,1,3]]
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying shortlex with Chenadec alphabet order:")
rounds = knuth_bendix(rules, order=shortlex_with_chenadec, reduce_crit_immediately=True, max_rounds=7, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")

# braid group < a,b,c | a^3 = b^2 = c >
rules = [[3,1,1,1,2,2], [3,1,1,1,3]]
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying shortlex with positive first order:")
rounds = knuth_bendix(rules, order=shortlex_with_positive_first, reduce_crit_immediately=True, max_rounds=7, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")

# braid group < a,b,c | a^3 = b^2 = c >
rules = [[3,1,1,1,2,2], [3,1,1,1,3]]
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying recursive path ordering:")
rounds = knuth_bendix(rules, order=recursive_path_order, reduce_crit_immediately=True, max_rounds=7, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")

# braid group < a,b,c | a^3 = b^2 = c >
rules = [[3,1,1,1,2,2], [3,1,1,1,3]]
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying recursive path ordering with Chenadec order:")
rounds = knuth_bendix(rules, order=recursive_path_order_with_chenadec, reduce_crit_immediately=True, max_rounds=7, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")



# braid group < a,b,c | a^3 = b^2 = c >
rules = [[3,1,1,1,2,2], [3,1,1,1,3]]
symmetrize_group_rule(rules,0)
symmetrize_group_rule(rules,1)
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying shortlex on symmetrized rules:")
rounds = knuth_bendix(rules, order=shortlex, reduce_crit_immediately=True, max_rounds=5, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")

# braid group < a,b,c | a^3 = b^2 = c >
rules = [[3,1,1,1,2,2], [3,1,1,1,3]]
symmetrize_group_rule(rules,0)
symmetrize_group_rule(rules,1)
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying recursive path ordering on symmetrized rules:")
rounds = knuth_bendix(rules, order=recursive_path_order, reduce_crit_immediately=True, max_rounds=5, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")

# braid group < a,b,c | a^3 = b^2 = c >
rules = [[3,1,1,1,2,2], [3,1,1,1,3]]
symmetrize_group_rule(rules,0)
symmetrize_group_rule(rules,1)
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying shortlex with Chenadec alphabet order on symmetrized rules:")
rounds = knuth_bendix(rules, order=shortlex_with_chenadec, reduce_crit_immediately=True, max_rounds=5, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))





Starting rules: [[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 3, -3], [2, -3, 3], [3, 1, 1, 1, 2, 2], [3, 1, 1, 1, 3]]
Trying shortlex:
Max rounds reached: 7
Making shortcuts and removing redundancies, then stopping.
Stopping because max rounds reached: 7
Terminated by reaching max_rounds.
Total number of rules: 114
[[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 3, -3], [2, -3, 3], [2, 2, 2, 3], [2, 3, 1, 1, 3], [2, 3, -1, -1, 3], [2, 1, 1, -1, 3], [3, -1, -1, 3, 1], [2, 3, 2, 2, 3], [2, 3, -2, 2], [2, -2, 3, 2], [3, -2, -1, 3, 2, -1], [3, -2, 1, 3, 2, 1], [2, 2, -3, -2], [2, -3, 1, -1, -1], [2, 1, -3, -1, -1], [2, -3, -1, -1, -3], [3, -1, -1, 2, 1, -2], [2, -3, 2, -2], [2, -3, -2, -2, -3], [2, -2, -2, -3], [3, 1, -2, -3, -1, -1, -2], [3, -1, -1, -1, -3], [3, 2, -1, -3, -2, -1], [3, 2, -1, -1, -2, 1], [3, 2, 1, -2, -2, 1, 2], [4, -2, 1, 2, 3, 2, 1, 2], [3, 2, -1, -2, -2, -1, 2], [4, -2, -1, 2, 3, 2, -1, 2], [5, -2, -1, 2, -1, 3, 2, -1, 2, -1], [5, -2, -1, 2, 1, 3, 2, -1, 2

In [19]:
# genus 2 orientable surface:
rules = [[8,1,2,3,4,-1,-2,-3,-4]]
# rules = [[2, 1, -1, ], [2,-1,1,], [2,2,-2,], [2,-2,2,], [2,3,-3,], [2,-3,3,], [6,1,2,3,-1,-2,-3]]
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying shortlex:")
rounds = knuth_bendix(rules, order=shortlex, reduce_crit_immediately=True, max_rounds=12, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("---------")

rules = [[8,1,2,3,4,-1,-2,-3,-4]]
# rules = [[2, 1, -1, ], [2,-1,1,], [2,2,-2,], [2,-2,2,], [2,3,-3,], [2,-3,3,], [6,1,2,3,-1,-2,-3]]
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying shortlex with Chenadec alphabet order:")
rounds = knuth_bendix(rules, order=shortlex_with_chenadec, reduce_crit_immediately=True, max_rounds=12, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("---------")

rules = [[8,1,2,3,4,-1,-2,-3,-4]]
# rules = [[2, 1, -1, ], [2,-1,1,], [2,2,-2,], [2,-2,2,], [2,3,-3,], [2,-3,3,], [6,1,2,3,-1,-2,-3]]
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying recursive path ordering:")
rounds = knuth_bendix(rules, order=recursive_path_order, reduce_crit_immediately=True, max_rounds=12, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))


Starting rules: [[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 3, -3], [2, -3, 3], [2, 4, -4], [2, -4, 4], [8, 1, 2, 3, 4, -1, -2, -3, -4]]
Trying shortlex:
Max rounds reached: 12
Making shortcuts and removing redundancies, then stopping.
Stopping because max rounds reached: 12
Terminated by reaching max_rounds.
Total number of rules: 44
[[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 3, -3], [2, -3, 3], [2, 4, -4], [2, -4, 4], [4, 4, -1, -2, -3, -3, -2, -1, 4], [4, 2, 3, 4, -1, -1, 4, 3, 2], [4, 4, 3, 2, 1, 1, 2, 3, 4], [4, 3, 4, -1, -2, -2, -1, 4, 3], [4, -4, -3, -2, -1, -1, -2, -3, -4], [5, -2, -1, 4, 3, 2, 3, 4, -1], [5, -3, -2, -1, 4, 3, 4, -1, -2], [4, -4, 1, 2, 3, 3, 2, 1, -4], [8, -3, -2, -1, 4, -2, -1, 4, 3, 4, -1, -2, 4, -1, -2], [8, -2, -1, 4, 3, -1, 4, 3, 2, 3, 4, -1, 3, 4, -1], [4, -2, -3, -4, 1, 1, -4, -3, -2], [11, -2, -1, 4, 3, -1, 4, 3, -1, 4, 3, 2, 3, 4, -1, 3, 4, -1, 3, 4, -1], [11, -3, -2, -1, 4, -2, -1, 4, -2, -1, 4, 3, 4, -1, -2, 4, -1, -2, 4, -1, -2], 

In [ ]:
# non-orientable surface group
rules = [[8,1,1,2,2,3,3,4,4]]
# rules = [[2, 1, -1, ], [2,-1,1,], [2,2,-2,], [2,-2,2,], [2,3,-3,], [2,-3,3,], [6,1,2,3,-1,-2,-3]]
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying shortlex:")
rounds = knuth_bendix(rules, order=shortlex, reduce_crit_immediately=True, max_rounds=15, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("---------")

rules = [[8,1,1,2,2,3,3,4,4]]
# rules = [[2, 1, -1, ], [2,-1,1,], [2,2,-2,], [2,-2,2,], [2,3,-3,], [2,-3,3,], [6,1,2,3,-1,-2,-3]]
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying shortlex with Chenadec alphabet order:")
rounds = knuth_bendix(rules, order=shortlex_with_chenadec, reduce_crit_immediately=True, max_rounds=15, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("---------")

rules = [[8,1,1,2,2,3,3,4,4]]
# rules = [[2, 1, -1, ], [2,-1,1,], [2,2,-2,], [2,-2,2,], [2,3,-3,], [2,-3,3,], [6,1,2,3,-1,-2,-3]]
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying shortlex with positive first order:")
rounds = knuth_bendix(rules, order=shortlex_with_positive_first, reduce_crit_immediately=True, max_rounds=15, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("---------")

rules = [[8,1,1,2,2,3,3,4,4]]
# rules = [[2, 1, -1, ], [2,-1,1,], [2,2,-2,], [2,-2,2,], [2,3,-3,], [2,-3,3,], [6,1,2,3,-1,-2,-3]]
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying recursive path ordering:")
rounds = knuth_bendix(rules, order=recursive_path_order, reduce_crit_immediately=True, max_rounds=15, print_progress=False, print_crit_pairs_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))

Starting rules: [[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 3, -3], [2, -3, 3], [2, 4, -4], [2, -4, 4], [8, 1, 1, 2, 2, 3, 3, 4, 4]]
Trying shortlex:
Max rounds reached: 15
Making shortcuts and removing redundancies, then stopping.
Stopping because max rounds reached: 15
Terminated by reaching max_rounds.
Total number of rules: 46
[[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 3, -3], [2, -3, 3], [2, 4, -4], [2, -4, 4], [4, 2, 3, 3, 4, -2, -1, -1, -4], [4, 1, 2, 2, 3, -1, -4, -4, -3], [4, -4, -4, -3, -3, 1, 1, 2, 2], [4, 2, 2, 3, 3, -1, -1, -4, -4], [4, 3, 3, 4, 4, -2, -2, -1, -1], [5, -2, -2, -1, -1, -4, 3, 3, 4], [7, 2, 2, 3, -2, -2, -1, -1, -1, -1, -4, -4, 3, 4, 4], [5, -1, -1, -4, -4, -3, 2, 2, 3], [5, -2, -1, -1, -4, -4, 2, 3, 3], [5, -3, -2, -2, -1, -1, 3, 4, 4], [7, 3, 4, 4, -1, -4, -4, -3, -3, -2, -2, -1, 2, 2, 3], [4, 3, 4, 4, 1, -3, -2, -2, -1], [4, 4, 1, 1, 2, -4, -3, -3, -2], [8, -2, -1, -1, -4, 1, 1, 2, 2, 2, 3, 3, -4, -3, -3], [11, -2, -1, -1, -4, 1, 1, 2, 

## Burnside groups

In [ ]:
def generate_group_words(words,number_gens=2,max_length=4, exact_length=False, remove_cyc_reducible=True, remove_shifts=True, remove_inverses=True):
  ''' generates cyclically reduced group words on number_gens group generators up to length max_length '''
  words.clear()
  for i in range(1,number_gens+1):
    words.append([i])
    words.append([-i])
  # print("Added words of length 1.")
  for length in range(2,max_length+1):
    # print("Processing length", length)
    new_words = []
    for word in words:
      # print(word)
      if len(word) == length-1:
        for i in range(1,number_gens+1):
          if word[-1] != i:
            new_words.append(word + [-i])
          if word[-1] != -i:
            new_words.append(word + [i])
    # print(new_words)
    words+= new_words

  # print(words)
  if exact_length == True:
    for word in words:
      # print("looking at", word)
      if len(word) != max_length:
        # print("removing", word)
        words.remove(word)
    if remove_inverses == True:
      for word in words:
        inverse_word = group_inverse(word)
        if inverse_word in words:
          # print("Removing inverse of", word, ":", inverse_word)
          words.remove(inverse_word)
          # print(len(words), "words left.")
    if remove_shifts == True:
      for word in words:
        for i in range(1,len(word)):
          shift_word = word[i:] + word[:i]
          if shift_word != word and shift_word in words:
            # print("Removing shift of", word, ":", shift_word)
            words.remove(shift_word)
    return 0

  if remove_cyc_reducible == True:
    i=0
    while i < len(words):
      if len(words[i]) > 2 and words[i][0]==-words[i][-1]:
        words.remove(words[i])
      else:
        i+=1

  if remove_inverses == True:
    for word in words:
      inverse_word = group_inverse(word)
      if inverse_word in words:
        # print("Removing inverse of", word, ":", inverse_word)
        words.remove(inverse_word)
        # print(len(words), "words left.")

  if remove_shifts == True:
    for word in words:
      for i in range(1,len(word)):
        shift_word = word[i:] + word[:i]
        if shift_word != word and shift_word in words:
          # print("Removing shift of", word, ":", shift_word)
          words.remove(shift_word)

  if remove_shifts and remove_inverses:
    for word in words:
      inverse_word = group_inverse(word)
      for i in range(0,len(word)):
        shift_word = inverse_word[i:] + inverse_word[:i]
        if shift_word != word and shift_word in words:
          words.remove(shift_word)

  return 0

def generate_all_words(words,number_gens=2,max_length=4, exact_length=False):
  words.clear()
  for i in range(1,number_gens+1):
    words.append([i])
    words.append([-i])
  # print("Added words of length 1.")
  for length in range(2,max_length+1):
    # print("Processing length", length)
    new_words = []
    for word in words:
      # print(word)
      if len(word) == length-1:
        for i in range(1,number_gens+1):
          new_words.append(word + [-i])
          new_words.append(word + [i])
    # print(new_words)
    words+= new_words

def presentation_to_rules(relators):
  rules = []
  for relator in relators:
    rules.append( [len(relator)] + relator )
  return rules

In [ ]:
def detect_inverses(words):
  for word in words:
    if group_inverse(word) in words:
      return True
  return False

def detect_duplicates(words):
  for word in words:
    if words.count(word) > 1:
      return True
  return False

def detect_free_inclusion(word,words):
  word_found = False
  cancellation_made = True
  while len(word) > 1 and cancellation_made:
    cancellation_made = False
    for i in range(len(word)-1):
      # print(word)
      if word[i]==-word[i+1]:
        cancellation_made = True
        word = word[:i] + ( word[i+2:] if i+2<len(word) else [] )
        if word == []:
          return True
        break

  while len(word) > 1 and word[0] == -word[-1]:
    # print(word, "-->", word[1:-1])
    word = word[1:-1]
    # print(word)
    if word == []:
      return True

  for i in range(len(word)):
    if word[i:]+word[:i] in words:
      return True

  word = group_inverse(word)
  for i in range(len(word)):
    if word[i:]+word[:i] in words:
      return True
  return False


In [ ]:
group_words = []
generate_group_words(group_words,number_gens=2,max_length=3,exact_length=False,remove_cyc_reducible=True,\
                     remove_shifts=True, remove_inverses=True)
# generate_group_words(group_words,number_gens=2,max_length=3,exact_length=False,remove_cyc_reducible=False,\
#                      remove_shifts=False, remove_inverses=False)
all_starting_relators = [word*3 for word in group_words]
all_starting_rules = presentation_to_rules(all_starting_relators)
# print(len(all_starting_rules))
rules = all_starting_rules

add_free_group_rules(rules)

print("B(2,3). Starting rules:", rules)
print("Trying shortlex:")
rounds = knuth_bendix(rules, order=shortlex, reduce_crit_immediately=True, max_rounds=4, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("---------")

group_words = []
generate_group_words(group_words,number_gens=2,max_length=3,exact_length=False,remove_cyc_reducible=True,\
                     remove_shifts=True, remove_inverses=True)
# generate_group_words(group_words,number_gens=2,max_length=3,exact_length=False,remove_cyc_reducible=False,\
#                      remove_shifts=False, remove_inverses=False)
all_starting_relators = [word*3 for word in group_words]
all_starting_rules = presentation_to_rules(all_starting_relators)
# print(len(all_starting_rules))
rules = all_starting_rules

add_free_group_rules(rules)

print("B(2,3). Starting rules:", rules)
print("Trying shortlex with Chenadec order:")
rounds = knuth_bendix(rules, order=shortlex_with_chenadec, reduce_crit_immediately=True, max_rounds=4, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("---------")

group_words = []
generate_group_words(group_words,number_gens=2,max_length=3,exact_length=False,remove_cyc_reducible=True,\
                     remove_shifts=True, remove_inverses=True)
# generate_group_words(group_words,number_gens=2,max_length=3,exact_length=False,remove_cyc_reducible=False,\
#                      remove_shifts=False, remove_inverses=False)
all_starting_relators = [word*3 for word in group_words]
all_starting_rules = presentation_to_rules(all_starting_relators)
# print(len(all_starting_rules))
rules = all_starting_rules

add_free_group_rules(rules)

print("B(2,3). Starting rules:", rules)
print("Trying shortlex with positive first:")
rounds = knuth_bendix(rules, order=shortlex_with_positive_first, reduce_crit_immediately=True, max_rounds=4, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("---------")

group_words = []
generate_group_words(group_words,number_gens=2,max_length=3,exact_length=False,remove_cyc_reducible=True,\
                     remove_shifts=True, remove_inverses=True)
# generate_group_words(group_words,number_gens=2,max_length=3,exact_length=False,remove_cyc_reducible=False,\
#                      remove_shifts=False, remove_inverses=False)
all_starting_relators = [word*3 for word in group_words]
all_starting_rules = presentation_to_rules(all_starting_relators)
# print(len(all_starting_rules))
rules = all_starting_rules

add_free_group_rules(rules)

print("B(2,3). Starting rules:", rules)
print("Trying recursive path order:")
rounds = knuth_bendix(rules, order=recursive_path_order, reduce_crit_immediately=True, max_rounds=4, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("---------")

group_words = []
generate_group_words(group_words,number_gens=2,max_length=3,exact_length=False,remove_cyc_reducible=True,\
                     remove_shifts=True, remove_inverses=True)
# generate_group_words(group_words,number_gens=2,max_length=3,exact_length=False,remove_cyc_reducible=False,\
#                      remove_shifts=False, remove_inverses=False)
all_starting_relators = [word*3 for word in group_words]
all_starting_rules = presentation_to_rules(all_starting_relators)
# print(len(all_starting_rules))
rules = all_starting_rules

add_free_group_rules(rules)

print("B(2,3). Starting rules:", rules)
print("Trying recursive path order with Chenadec order:")
rounds = knuth_bendix(rules, order=recursive_path_order_with_chenadec, reduce_crit_immediately=True, max_rounds=4, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("---------")

group_words = []
generate_group_words(group_words,number_gens=2,max_length=3,exact_length=False,remove_cyc_reducible=True,\
                     remove_shifts=True, remove_inverses=True)
# generate_group_words(group_words,number_gens=2,max_length=3,exact_length=False,remove_cyc_reducible=False,\
#                      remove_shifts=False, remove_inverses=False)
all_starting_relators = [word*3 for word in group_words]
all_starting_rules = presentation_to_rules(all_starting_relators)
# print(len(all_starting_rules))
rules = all_starting_rules

add_free_group_rules(rules)

print("B(2,3). Starting rules:", rules)
print("Trying recursive path order with positive first order:")
rounds = knuth_bendix(rules, order=recursive_path_order_with_positive_first, reduce_crit_immediately=True, max_rounds=4, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("---------")

B(2,3). Starting rules: [[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [3, 1, 1, 1], [3, 2, 2, 2], [6, 1, 1, 1, 1, 1, 1], [6, 1, -2, 1, -2, 1, -2], [6, 1, 2, 1, 2, 1, 2], [6, 2, 2, 2, 2, 2, 2], [9, 1, 1, 1, 1, 1, 1, 1, 1, 1], [9, 1, 1, -2, 1, 1, -2, 1, 1, -2], [9, 1, 1, 2, 1, 1, 2, 1, 1, 2], [9, 1, -2, -2, 1, -2, -2, 1, -2, -2], [9, 1, 2, 2, 1, 2, 2, 1, 2, 2], [9, 2, 2, 2, 2, 2, 2, 2, 2, 2]]
Trying shortlex:
Max rounds reached: 4
Making shortcuts and removing redundancies, then stopping.
Stopping because max rounds reached: 4
Terminated by reaching max_rounds.
Total number of rules: 26
[[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 2, 2, -2], [2, 1, 1, -1], [2, -1, -1, 1], [4, -2, 1, 2, -1, -1, -2, 1, 2], [4, -2, -1, 2, 1, -1, 2, 1, -2], [4, 1, 2, 1, -2, -2, -1, 2], [4, -2, -1, 2, -1, 2, 1, -2], [4, 2, 1, -2, 1, -2, -1, 2], [2, -2, -2, 2], [4, 1, 2, -1, -2, -1, -2, 1, 2], [4, 1, -2, -1, 2, -1, 2, 1, -2], [3, -2, 1, -2, -1, 2, -1], [3, 2, -1, 2, 1, -2, 1], [4, -2, 1, 2, 1, 2, -1

In [ ]:
group_words = []
generate_group_words(group_words,number_gens=2,max_length=4,exact_length=False,remove_cyc_reducible=True,\
                     remove_shifts=True, remove_inverses=True)
# generate_group_words(group_words,number_gens=2,max_length=3,exact_length=False,remove_cyc_reducible=False,\
#                      remove_shifts=False, remove_inverses=False)
all_starting_relators = [word*4 for word in group_words]
all_starting_rules = presentation_to_rules(all_starting_relators)
print(len(all_starting_rules))
rules = all_starting_rules

add_free_group_rules(rules)

print("B(2,4). Starting rules:", rules)
print("Trying recursive path order:")
rounds = knuth_bendix(rules, order=recursive_path_order, reduce_crit_immediately=True, max_rounds=4, print_progress=True)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("---------")
